# Shared Setup

Shared constants and helper functions for the MobileNetV3-small notebook suite.

In [ ]:
from __future__ import annotations

import random
import subprocess
from pathlib import Path

import numpy as np

try:
    import tensorflow as tf
except Exception:
    tf = None

NOTEBOOK_OVERRIDES = globals().get('NOTEBOOK_OVERRIDES', {})


def override(name: str, default: object) -> object:
    return NOTEBOOK_OVERRIDES.get(name, default)


NOTEBOOK_TEST_MODE = bool(override('NOTEBOOK_TEST_MODE', False))
DATA_ROOT = Path(str(override('DATA_ROOT', Path.cwd() / 'data')))
ROOT = Path(str(override('ROOT', Path.cwd())))
RAW_DATA_ROOT = Path(str(override('RAW_DATA_ROOT', DATA_ROOT / 'raw_dataset')))
GENERATED_SPLITS_ROOT = Path(str(override('GENERATED_SPLITS_ROOT', ROOT / 'generated_splits')))
PROCESSED_ROI_ROOT = Path(str(override('PROCESSED_ROI_ROOT', DATA_ROOT / 'processed_hsv_lab_threshold_roi_224')))
CANONICAL_PROCESSING_SUMMARY_PATH = Path(
    str(override('CANONICAL_PROCESSING_SUMMARY_PATH', PROCESSED_ROI_ROOT / 'processing_summary.csv'))
)
TRAINING_OUTPUTS_ROOT = Path(str(override('TRAINING_OUTPUTS_ROOT', ROOT / 'training_outputs')))

LABEL_ORDER = ['fresh', 'not fresh', 'spoiled']
RUN_SEEDS = [42, 123, 2026]
TARGET_SIZE = (224, 224)
INPUT_SHAPE = (224, 224, 3)
BATCH_SIZE = int(override('BATCH_SIZE', 32))
EPOCHS_HEAD = int(override('EPOCHS_HEAD', 8))
EPOCHS_FINE = int(override('EPOCHS_FINE', 20))
HEAD_LR = float(override('HEAD_LR', 5e-4))
FINE_TUNE_LR = float(override('FINE_TUNE_LR', 1e-5))
FINE_TUNE_FRACTION = float(override('FINE_TUNE_FRACTION', 0.25))
REQUIRED_TRAINING_GPU_SUBSTRING = 'RTX 4050'


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    if tf is not None:
        tf.keras.utils.set_random_seed(seed)


def enforce_training_gpu(required_name_substring: str = REQUIRED_TRAINING_GPU_SUBSTRING) -> list[str]:
    if NOTEBOOK_TEST_MODE or bool(override('SKIP_GPU_CHECK', False)):
        return []

    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
            check=True,
            capture_output=True,
            text=True,
        )
    except Exception as exc:
        raise RuntimeError('Unable to verify NVIDIA GPU availability via nvidia-smi.') from exc

    gpu_names = [line.strip() for line in result.stdout.splitlines() if line.strip()]
    if not gpu_names:
        raise RuntimeError('No NVIDIA GPU detected. Training must run on the laptop RTX 4050.')

    if tf is None:
        raise RuntimeError('TensorFlow is unavailable, so GPU-backed training cannot start.')

    tensorflow_gpus = tf.config.list_physical_devices('GPU')
    if not tensorflow_gpus:
        raise RuntimeError('TensorFlow does not currently see any GPU devices. Fix CUDA/TensorFlow GPU support before training.')

    normalized_required = required_name_substring.lower()
    if not any(normalized_required in gpu_name.lower() for gpu_name in gpu_names):
        raise RuntimeError(
            f'Required GPU containing {required_name_substring!r} was not found. Detected GPUs: {gpu_names}'
        )
    return gpu_names


for path in (GENERATED_SPLITS_ROOT, TRAINING_OUTPUTS_ROOT):
    ensure_dir(path)
